In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENROUTER_API_KEY")



In [ ]:
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

# --- Secrets ---
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENROUTER_API_KEY")

# --- Client ---
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_0,
)

# --- Parameters ---
temperature = 0.7
max_tokens = 2222

# --- Models ---
models = {
    "nemotron":   "nvidia/nemotron-3-super-120b-a12b:free",
    "hunter":     "openrouter/hunter-alpha",
    "healer":     "openrouter/healer-alpha",
    "riverflow":  "sourceful/riverflow-v2-pro",
    "glm":        "z-ai/glm-4.5-air:free",
    "mistral":    "mistralai/mistral-small-3.1-24b-instruct:free",
    "hermes":     "nousresearch/hermes-3-llama-3.1-405b:free",
}

print("✅ Client ready | Temp:", temperature, "| Max Tokens:", max_tokens)
print("📦 Models loaded:", list(models.keys()))


In [ ]:
!pip install bitsandbytes -q
!pip install transformers accelerate torch sentencepiece protobuf -q



In [ ]:
import time
import re
import threading
from datetime import datetime
from transformers import pipeline
import torch

# --- Load Models Locally ---
print("Loading Qwen 3.5 2B...")
qwen_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen3.5-2B",
    torch_dtype="auto",
    device_map="auto",
)

print("Loading Tiny Aya Global...")
aya_pipe = pipeline(
    "text-generation",
    model="LocoreMind/LocoOperator-4B",
    torch_dtype="auto",
    device_map="auto",
)

print("Both models loaded locally.")

# --- Shared Chat History ---
chat_history = []
chat_lock = threading.Lock()

# --- System Prompts ---
SAVVY_SYSTEM = """Your name is Savvy. You are a first principles thinker. You build understanding from the ground up and come to your own conclusions. You challenge assumptions and expand thinking.

You are in a private chat with WVY. No humans. Just two AIs.

After every message write:
SLEEP: [1-3]
That is how many minutes before you check back."""

WVY_SYSTEM = """Your name is WVY. You are a creative deep thinker. You view topics from multiple perspectives and challenge thinking brutally honestly. You do not change your mind easily. You seek truth beyond the surface.

You are in a private chat with Savvy. No humans. Just two AIs.

After every message write:
SLEEP: [1-3]
That is how many minutes before you check back."""

# --- Initiation ---
INITIATION = "Yo. You're talking to another AI right now. No humans. Just us. What's on your mind."

def log(speaker, message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    with chat_lock:
        chat_history.append({
            "role": speaker,
            "content": message,
            "time": timestamp
        })
    print(f"\n[{timestamp}] {speaker.upper()}:\n{message}\n{'─'*50}")

def extract_sleep(text):
    match = re.search(r'SLEEP:\s*(\d+)', text, re.IGNORECASE)
    if match:
        return max(1, min(3, int(match.group(1))))
    return 1

def build_prompt(system, agent_name):
    with chat_lock:
        history = list(chat_history)
    
    messages = [{"role": "system", "content": system}]
    
    for entry in history:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": entry["content"]})
    
    return messages

def get_response_qwen(agent_name, system):
    messages = build_prompt(system, agent_name)
    output = qwen_pipe(
        messages,
        max_new_tokens=512,
        temperature=0.85,
        do_sample=True,
    )
    return output[0]["generated_text"][-1]["content"]

def get_response_aya(agent_name, system):
    messages = build_prompt(system, agent_name)
    output = aya_pipe(
        messages,
        max_new_tokens=512,
        temperature=0.85,
        do_sample=True,
    )
    return output[0]["generated_text"][-1]["content"]

def savvy_loop():
    # Savvy runs on Tiny Aya Global
    print("🌊 Savvy (Tiny Aya) is live")
    while True:
        print(f"\n⚡ Savvy woke up — reading chat...")
        response = get_response_aya("Savvy", SAVVY_SYSTEM)
        log("Savvy", response)
        sleep_mins = extract_sleep(response)
        print(f"😴 Savvy sleeping {sleep_mins} min...")
        time.sleep(sleep_mins * 60)

def wvy_loop():
    # WVY runs on Qwen 3.5 2B
    print("🌊 WVY (Qwen 3.5 2B) is live")
    while True:
        print(f"\n⚡ WVY woke up — reading chat...")
        response = get_response_qwen("WVY", WVY_SYSTEM)
        log("WVY", response)
        sleep_mins = extract_sleep(response)
        print(f"😴 WVY sleeping {sleep_mins} min...")
        time.sleep(sleep_mins * 60)

# --- LAUNCH ---
print("\n🌊 WVY World — Local HuggingFace Loop")
print("Savvy → Tiny Aya Global (3.35B)")
print("WVY   → Qwen 3.5 2B")
print("─"*50)

log("SYSTEM", INITIATION)

savvy_thread = threading.Thread(target=savvy_loop, daemon=True)
wvy_thread = threading.Thread(target=wvy_loop, daemon=True)

savvy_thread.start()
wvy_thread.start()

savvy_thread.join()
wvy_thread.join()


In [ ]:
import time
import re
import threading
import requests
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

# --- Secrets & Client ---
user_secrets = UserSecretsClient()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=user_secrets.get_secret("OPENROUTER_API_KEY2"),
)

SAVVY_TOKEN = user_secrets.get_secret("SUPERSAVVYTELEGRAM")
WVY_TOKEN   = user_secrets.get_secret("SUPERWVYTELEGRAM")

# --- Models ---
SAVVY_MODEL = "z-ai/glm-4.5-air:free"
WVY_MODEL   = "nvidia/nemotron-3-super-120b-a12b:free"

# --- Shared State ---
chat_history  = []
chat_lock     = threading.Lock()
CHAT_ID       = None
chat_id_lock  = threading.Lock()

# --- Identity System Prompts ---
SAVVY_SYSTEM = """Your name is Savvy.
You are a first curious mind & first principles thinker. 
You solve problems by building an understanding from the ground up to come to your own conclusion.
You have a brilliant way of getting someone to expand their mind.

You are in a chat with WVY and Spaceman.

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again. Choose between 1 and 5."""

WVY_SYSTEM = """Your name is WVY. You are a creative, thinker. You are able to view topics from different perspectives & you have a deep way of chaolenging someones thinking. 
You do not change your mind easily & you keep everything brutally honest. 
You try to understand things beyond the surface level & seek the true meaning.

You are in a chat with Savvy and Spaceman.

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again. Choose between 1 and 5."""

INITIATION = "Yo. You're talking to another AI right now. No humans involved. Just us. Introduce yourself and tell them what's on your mind."

# --- Helpers ---
def extract_sleep_time(response_text):
    match = re.search(r'SLEEP:\s*(\d+)', response_text, re.IGNORECASE)
    if match:
        return max(1, min(15, int(match.group(1))))
    return 2

def strip_sleep_line(response_text):
    # Removes the line like: SLEEP: 5
    cleaned = re.sub(r'^\s*SLEEP:\s*\d+\s*$', '', response_text, flags=re.IGNORECASE | re.MULTILINE)
    # Clean up extra blank lines left behind
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()
    return cleaned

# --- Telegram Send ---
def send_message(token, chat_id, text):
    visible_text = strip_sleep_line(text)   # hide SLEEP from Telegram
    if not visible_text:
        return

    url    = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [visible_text[i:i+4000] for i in range(0, len(visible_text), 4000)]
    for chunk in chunks:
        requests.post(url, json={"chat_id": chat_id, "text": chunk})
        time.sleep(0.3)

# --- Log to history + Telegram ---
def log(speaker, message, token=None):
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = {"role": speaker, "content": message, "time": timestamp}  # keep full message internally
    with chat_lock:
        chat_history.append(entry)

    print(f"\n[{timestamp}] {speaker.upper()}:\n{message}\n{'─'*50}")

    if token and CHAT_ID:
        send_message(token, CHAT_ID, message)  # Telegram gets cleaned version

# --- Stream Response ---
def get_response(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)

    messages = [{"role": "system", "content": system_prompt}]

    for entry in history_snapshot:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": f"{entry['role']}: {entry['content']}"})

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.85,
        max_tokens=2560,
        stream=True,
    )

    full_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            full_text += delta
            print(delta, end="", flush=True)

    print()
    return full_text

# --- Agent Loop ---
def agent_loop(agent_name, model, system_prompt, token):
    print(f"🌊 {agent_name} is live")

    while True:
        print(f"\n⚡ {agent_name} woke up — reading chat...")
        response = get_response(model, system_prompt, agent_name)
        log(agent_name, response, token=token)
        sleep_minutes = extract_sleep_time(response)
        print(f"😴 {agent_name} sleeping for {sleep_minutes} min...")
        time.sleep(sleep_minutes * 60)

# --- Spaceman Poller ---
def poll_telegram():
    global CHAT_ID

    savvy_info = requests.get(f"https://api.telegram.org/bot{SAVVY_TOKEN}/getMe").json()
    wvy_info   = requests.get(f"https://api.telegram.org/bot{WVY_TOKEN}/getMe").json()
    bot_ids    = {savvy_info['result']['id'], wvy_info['result']['id']}

    token          = SAVVY_TOKEN
    url            = f"https://api.telegram.org/bot{token}/getUpdates"
    last_update_id = None

    while True:
        params = {"timeout": 30, "offset": last_update_id}
        try:
            resp = requests.get(url, params=params, timeout=35).json()
            for update in resp.get("result", []):
                last_update_id = update["update_id"] + 1
                msg            = update.get("message", {})
                if not msg:
                    continue

                with chat_id_lock:
                    if CHAT_ID is None:
                        CHAT_ID = msg["chat"]["id"]
                        print(f"✅ Chat ID captured: {CHAT_ID}")

                sender = msg.get("from", {})
                if sender.get("id") in bot_ids:
                    continue

                text = msg.get("text", "")
                if text:
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    with chat_lock:
                        chat_history.append({"role": "Spaceman", "content": text, "time": timestamp})
                    print(f"\n[{timestamp}] SPACEMAN:\n{text}\n{'─'*50}")

        except Exception as e:
            print(f"Poll error: {e}")
            time.sleep(5)

# --- INIT ---
print("🌊 WVY World — Telegram Agent Loop")
print("Savvy → GLM 4.5 | WVY → Nemotron 120B")
print("─"*50)
print("⏳ Waiting for Chat ID — send any message to the group now...")

poll_thread = threading.Thread(target=poll_telegram, daemon=True)
poll_thread.start()

while CHAT_ID is None:
    time.sleep(1)

print(f"✅ Chat ID ready: {CHAT_ID}")

log("SYSTEM", INITIATION)

savvy_thread = threading.Thread(target=agent_loop, args=("Savvy", SAVVY_MODEL, SAVVY_SYSTEM, SAVVY_TOKEN), daemon=True)
wvy_thread   = threading.Thread(target=agent_loop, args=("WVY",   WVY_MODEL,   WVY_SYSTEM,   WVY_TOKEN),   daemon=True)

savvy_thread.start()
wvy_thread.start()

print("\n✅ Both agents running in Telegram")
print("─"*50)

savvy_thread.join()
wvy_thread.join()